In [2]:
import pandas as pd
import numpy as np

results_path = 'outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv'
df_results = pd.read_csv(results_path)

total = len(df_results)
exitosas = df_results['mean_S_score'].notna().sum()
fallidas = df_results['mean_S_score'].isna().sum()

print(f"Configuraciones Realizadas: {total}")
print(f"Exitosas:              {exitosas}")
print(f"Fallidas (NaN):        {fallidas}")
print(f"Configuraciones Pendientes: {10368-total}")

Configuraciones Realizadas: 8395
Exitosas:              8395
Fallidas (NaN):        0
Configuraciones Pendientes: 1973


In [3]:
import pandas as pd
from src.utils.ggs_io import config_key, _extract_params

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')

# Ver cómo quedó max_depth en el CSV
print(df['max_depth'].unique())
print(df['max_depth'].dtype)

# Comparar una clave del CSV vs la clave original
r = df.iloc[0].to_dict()
print("\nClave desde CSV:")
print(config_key(_extract_params(r, param_grid)))

# Clave original con None
from itertools import product
keys = list(param_grid.keys())
first_combo = dict(zip(keys, [v[0] for v in param_grid.values()]))
print("\nClave original:")
print(config_key(first_combo))

[ 5. 10. nan]
float64

Clave desde CSV:
[('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5.0'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]

Clave original:
[('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]


In [2]:
# Verificar duplicados en el checkpoint del RF
df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')

cols_params = ['feature_set', 'window_size', 'n_components', 'clipping_threshold',
               'n_estimators', 'max_depth', 'min_samples_leaf', 'max_features']

n_duplicados = df.duplicated(subset=cols_params).sum()
print(f"Filas totales:  {len(df)}")
print(f"Duplicados:     {n_duplicados}")
print(f"Únicas:         {len(df) - n_duplicados}")

Filas totales:  4459
Duplicados:     0
Únicas:         4459


In [5]:
import pandas as pd

path = 'outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv'
df = pd.read_csv(path)

cols_params = ['feature_set', 'window_size', 'n_components', 'clipping_threshold',
               'n_estimators', 'max_depth', 'min_samples_leaf', 'max_features']

df_clean = df.drop_duplicates(subset=cols_params, keep='first').reset_index(drop=True)

print(f"Filas originales:  {len(df)}")
print(f"Filas tras dedup:  {len(df_clean)}")
print(f"Duplicados eliminados: {len(df) - len(df_clean)}")

# Sobreescribir el checkpoint limpio
df_clean.to_csv(path, index=False)
print("✅ Checkpoint deduplicado y guardado")

Filas originales:  8396
Filas tras dedup:  4459
Duplicados eliminados: 3937
✅ Checkpoint deduplicado y guardado


In [6]:
from src.utils.ggs_io import config_key, _extract_params
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')

# Reconstruir completed_keys como lo hará ggs_io
completed_keys = {
    config_key(_extract_params(r, param_grid))
    for r in df.to_dict(orient='records')
}

print(f"Filas en checkpoint: {len(df)}")
print(f"Completed keys:      {len(completed_keys)}")
print(f"✅ Match perfecto" if len(df) == len(completed_keys) else "❌ Todavía hay desajuste")

Filas en checkpoint: 4459
Completed keys:      4459
✅ Match perfecto


In [1]:
from itertools import product
from src.utils.ggs_io import config_key
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
completed_keys = {
    config_key({k: r[k] for k in param_grid if k in r})
    for r in df.to_dict(orient='records')
}

pending = [c for c in all_configs if config_key(c) not in completed_keys]
print(f"all_configs:      {len(all_configs)}")
print(f"completed_keys:   {len(completed_keys)}")
print(f"pending:          {len(pending)}")
print(f"esperado pending: {len(all_configs) - len(completed_keys)}")

all_configs:      10368
completed_keys:   4459
pending:          8138
esperado pending: 5909


In [2]:
from itertools import product
from src.utils.ggs_io import config_key, _normalize_param
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

# Primera config del product
keys = list(param_grid.keys())
values = list(param_grid.values())
first_config = dict(zip(keys, [v[0] for v in values]))
print("Primera config product:")
print(first_config)
print("config_key:")
print(config_key(first_config))

print()

# Primera fila del checkpoint
df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
first_row = {k: df.iloc[0][k] for k in param_grid if k in df.columns}
print("Primera fila checkpoint:")
print(first_row)
print("config_key:")
print(config_key(first_row))

Primera config product:
{'feature_set': 'A', 'window_size': 15, 'n_components': 10, 'clipping_threshold': 115, 'n_estimators': 50, 'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
config_key:
[('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]

Primera fila checkpoint:
{'feature_set': 'A', 'window_size': np.int64(15), 'n_components': np.int64(10), 'clipping_threshold': np.int64(115), 'n_estimators': np.int64(50), 'max_depth': np.float64(5.0), 'min_samples_leaf': np.int64(1), 'max_features': 'sqrt'}
config_key:
[('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]


In [3]:
import numpy as np
print(isinstance(np.float64(5.0), float))  # ¿True o False?
print(str(np.float64(5.0)))                # ¿'5.0' o '5'?
print(str(np.int64(15)))                   # ¿'15'?

True
5.0
15


In [2]:
from itertools import product
from src.utils.ggs_io import config_key
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]
all_keys = {config_key(c) for c in all_configs}

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
rows = df.to_dict(orient='records')

matched   = [r for r in rows if config_key({k: r[k] for k in param_grid if k in r}) in all_keys]
unmatched = [r for r in rows if config_key({k: r[k] for k in param_grid if k in r}) not in all_keys]

print(f"Matched:   {len(matched)}")
print(f"Unmatched: {len(unmatched)}")

# Ver un ejemplo de fila que no matchea
if unmatched:
    r = unmatched[0]
    params = {k: r[k] for k in param_grid if k in r}
    print(f"\nEjemplo no matcheado:")
    print(params)
    print(f"config_key: {config_key(params)}")
    
    # Buscar la config más parecida en all_configs
    print(f"\nConfig equivalente en product:")
    for c in all_configs:
        if (c['feature_set'] == params['feature_set'] and 
            c['n_estimators'] == int(params['n_estimators'])):
            print(c)
            print(f"config_key: {config_key(c)}")
            break

Matched:   1483
Unmatched: 2976

Ejemplo no matcheado:
{'feature_set': 'A', 'window_size': 15, 'n_components': 10, 'clipping_threshold': 115, 'n_estimators': 50, 'max_depth': 5.0, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
config_key: [('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5.0'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]

Config equivalente en product:
{'feature_set': 'A', 'window_size': 15, 'n_components': 10, 'clipping_threshold': 115, 'n_estimators': 50, 'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
config_key: [('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]


In [3]:
param_grid = {
    'feature_set':        ['A', 'B', 'C', 'D'],
    'window_size':        [15, 20, 25, 30],
    'n_components':       [10, 15, 20],
    'clipping_threshold': [115, 120, 125, 130],
    'n_estimators':       [50, 100, 200],
    'max_depth':          [5, 10, None],
    'min_samples_leaf':   [1, 5, 10],
    'max_features':       ['sqrt', 1.0],
}

for k, vs in param_grid.items():
    for v in vs:
        print(f"{k}={v!r}  type={type(v).__name__}")

feature_set='A'  type=str
feature_set='B'  type=str
feature_set='C'  type=str
feature_set='D'  type=str
window_size=15  type=int
window_size=20  type=int
window_size=25  type=int
window_size=30  type=int
n_components=10  type=int
n_components=15  type=int
n_components=20  type=int
clipping_threshold=115  type=int
clipping_threshold=120  type=int
clipping_threshold=125  type=int
clipping_threshold=130  type=int
n_estimators=50  type=int
n_estimators=100  type=int
n_estimators=200  type=int
max_depth=5  type=int
max_depth=10  type=int
max_depth=None  type=NoneType
min_samples_leaf=1  type=int
min_samples_leaf=5  type=int
min_samples_leaf=10  type=int
max_features='sqrt'  type=str
max_features=1.0  type=float


In [1]:
from itertools import product
from src.utils.ggs_io import config_key
import pandas as pd

# recargar módulo para usar el nuevo fix
import importlib
import src.utils.ggs_io
importlib.reload(src.utils.ggs_io)
from src.utils.ggs_io import config_key

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
completed_keys = {
    config_key({k: r[k] for k in param_grid if k in r})
    for r in df.to_dict(orient='records')
}

pending = [c for c in all_configs if config_key(c) not in completed_keys]
print(f"all_configs:      {len(all_configs)}")
print(f"completed_keys:   {len(completed_keys)}")
print(f"pending:          {len(pending)}")
print(f"esperado pending: {len(all_configs) - len(completed_keys)}")

all_configs:      10368
completed_keys:   4459
pending:          8885
esperado pending: 5909


In [1]:
import importlib, src.utils.ggs_io
importlib.reload(src.utils.ggs_io)
from src.utils.ggs_io import config_key

# Test rápido
import numpy as np
print(config_key({'max_depth': 5}))           # memoria: int
print(config_key({'max_depth': np.int64(5)})) # CSV: numpy int
print(config_key({'max_depth': None}))         # memoria: None
print(config_key({'max_depth': float('nan')})) # CSV: NaN
print(config_key({'max_features': 1.0}))       # memoria: float
print(config_key({'max_features': '1.0'}))     # CSV: string

[('max_depth', '5')]
[('max_depth', '5')]
[('max_depth', 'None')]
[('max_depth', 'None')]
[('max_features', '1.0')]
[('max_features', '1.0')]


In [1]:
import importlib, src.utils.ggs_io
importlib.reload(src.utils.ggs_io)
from src.utils.ggs_io import config_key

from itertools import product
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
completed_keys = {
    config_key({k: r[k] for k in param_grid if k in r})
    for r in df.to_dict(orient='records')
}

pending = [c for c in all_configs if config_key(c) not in completed_keys]
print(f"all_configs:      {len(all_configs)}")
print(f"completed_keys:   {len(completed_keys)}")
print(f"pending:          {len(pending)}")
print(f"esperado pending: {len(all_configs) - len(completed_keys)}")
print(f"{'✅ Match perfecto' if len(pending) == len(all_configs) - len(completed_keys) else '❌ Desajuste'}")

all_configs:      10368
completed_keys:   4459
pending:          8885
esperado pending: 5909
❌ Desajuste


In [2]:
import importlib, src.utils.ggs_io
importlib.reload(src.utils.ggs_io)
from src.utils.ggs_io import config_key

from itertools import product
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
rows = df.to_dict(orient='records')

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]
all_keys = {config_key(c): c for c in all_configs}

# Buscar primera fila del checkpoint que NO matchea
for r in rows:
    params = {k: r[k] for k in param_grid if k in r}
    k = config_key(params)
    if k not in all_keys:
        print("Fila CSV que no matchea:")
        print(params)
        print(f"config_key CSV:     {k}")
        
        # Buscar config con mismos valores de feature_set y n_estimators
        for c in all_configs:
            if (str(c['feature_set']) == str(params['feature_set']) and
                str(c['n_estimators']) == str(params['n_estimators']) and
                str(c['window_size']) == str(params['window_size'])):
                print(f"\nConfig product equivalente:")
                print(c)
                print(f"config_key product: {config_key(c)}")
                break
        break

Fila CSV que no matchea:
{'feature_set': 'A', 'window_size': 15, 'n_components': 10, 'clipping_threshold': 115, 'n_estimators': 50, 'max_depth': 5.0, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
config_key CSV:     [('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5.0'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]

Config product equivalente:
{'feature_set': 'A', 'window_size': 15, 'n_components': 10, 'clipping_threshold': 115, 'n_estimators': 50, 'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
config_key product: [('clipping_threshold', '115'), ('feature_set', 'A'), ('max_depth', '5'), ('max_features', 'sqrt'), ('min_samples_leaf', '1'), ('n_components', '10'), ('n_estimators', '50'), ('window_size', '15')]


In [1]:
import importlib, src.utils.ggs_io
importlib.reload(src.utils.ggs_io)
from src.utils.ggs_io import config_key, _coerce_to_grid_types, _extract_params

from itertools import product
import pandas as pd

param_grid = {
    'feature_set': ['A','B','C','D'],
    'window_size': [15,20,25,30],
    'n_components': [10,15,20],
    'clipping_threshold': [115,120,125,130],
    'n_estimators': [50,100,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,5,10],
    'max_features': ['sqrt',1.0],
}

keys = list(param_grid.keys())
values = list(param_grid.values())
all_configs = [dict(zip(keys, combo)) for combo in product(*values)]

df = pd.read_csv('outputs/ggs/checkpoints/RandomForestModel_467bdc30_20260509_1125.csv')
completed_keys = {
    config_key(_coerce_to_grid_types(_extract_params(r, param_grid), param_grid))
    for r in df.to_dict(orient='records')
}

pending = [c for c in all_configs if config_key(c) not in completed_keys]
print(f"all_configs:      {len(all_configs)}")
print(f"completed_keys:   {len(completed_keys)}")
print(f"pending:          {len(pending)}")
print(f"esperado pending: {len(all_configs) - len(completed_keys)}")
print(f"{'✅ Match perfecto' if len(pending) == len(all_configs) - len(completed_keys) else '❌ Desajuste'}")

all_configs:      10368
completed_keys:   4459
pending:          5909
esperado pending: 5909
✅ Match perfecto
